In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from keras.preprocessing import image
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, concatenate
from keras.callbacks import ModelCheckpoint
from PIL import Image
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

2024-06-11 22:13:14.987924: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-06-11 22:13:20.106651: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-11 22:13:26.869516: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# Load environmental data from CSV
environmental_data = pd.read_csv('Train Data/Disease_Data.csv')

# Extract image file paths and corresponding labels
image_paths = environmental_data['image_file'].values
labels = environmental_data['disease'].values

In [ ]:
num_classes = 6
# Load and preprocess image data
image_data = []
for img_path in image_paths:
    img = Image.open(img_path)
    img = img.resize((80, 80))  # Resize image to target size
    img_array = np.array(img) / 255.0  # Normalize pixel values to [0, 1]
    image_data.append(img_array)


: 

In [ ]:
image_data = np.array(image_data)

In [ ]:
# Load environmental features
environmental_features = environmental_data.drop(columns=['image_file', 'disease']).values

In [ ]:
# Define the image input
image_input = Input(shape=(120, 120, 3))  # Assuming RGB images

In [ ]:
x = Conv2D(32, (3, 3), activation='relu')(image_input)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(128, (3, 3), activation='relu')(x)  # Add another convolutional layer with 128 filters
x = MaxPooling2D((2, 2))(x)
x = Conv2D(256, (3, 3), activation='relu')(x)  # Add another convolutional layer with 128 filters
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
# Define the environmental data input
environmental_input = Input(shape=(environmental_features.shape[1],))
# Merge image and environmental data
combined = concatenate([x, environmental_input])

In [ ]:
# Add fully connected layers for classification
x = Dense(256, activation='relu')(combined)
output = Dense(6, activation='softmax')(x)  # Output layer with softmax activation for multi-class classification

In [ ]:
# Create the model
model = Model(inputs=[image_input, environmental_input], outputs=output)
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Split data into training and validation sets
X_train_img, X_val_img, y_train, y_val = train_test_split(image_data, labels, test_size=0.2, random_state=42)
X_train_env, X_val_env = train_test_split(environmental_features, test_size=0.2, random_state=42)

In [ ]:
# Standardize environmental data
scaler = StandardScaler()
X_train_env_scaled = scaler.fit_transform(X_train_env)
X_val_env_scaled = scaler.transform(X_val_env)

In [ ]:
# Define a checkpoint callback to save the best model during training
checkpoint = ModelCheckpoint('best_model.h5', monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')

# Encode categorical labels into numerical values
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

# Convert encoded labels to one-hot encoded format
num_classes = len(label_encoder.classes_)
y_train_one_hot = to_categorical(y_train_encoded, num_classes)
y_val_one_hot = to_categorical(y_val_encoded, num_classes)

In [ ]:
# Train the model
history = model.fit([X_train_img, X_train_env_scaled], y_train_one_hot, epochs=15, batch_size=32, validation_data=([X_val_img, X_val_env_scaled], y_val_one_hot), callbacks=[checkpoint])

In [ ]:
# Evaluate the model


y_pred = model.predict([X_val_img, X_val_env_scaled])
y_pred_labels = np.argmax(y_pred, axis=1)
#print(classification_report(y_val, y_pred_labels, target_names="Brown_Rust"))
# Convert validation set labels from string to numerical values
y_val_encoded = label_encoder.transform(y_val)

In [ ]:
# Extract training history
train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs = range(1, len(train_loss) + 1)

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, 'bo-', label='Training loss')
plt.plot(epochs, val_loss, 'ro-', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_acc, 'bo-', label='Training accuracy')
plt.plot(epochs, val_acc, 'ro-', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate classification report
print(classification_report(y_val_encoded, y_pred_labels, target_names=label_encoder.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val_encoded, y_pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Save the trained model
model.save('New_model_256_15_32.h5')